# UnboxAI: Search & Recommendations with BehaviorGPT

**BehaviorGPT** is trained on long sequences of things people actually did — viewed this, added that, bought the other — and predicts what comes next. Same idea as a language model predicting the next word, except the sequence is actions instead of text.

This notebook serves as an interactive environment and testing playground for the UnboxAI **BehaviorGPT**.


### Prerequisites for Client Interaction

- **API Key & Authentication:** Access to the Unbox API requires a valid API key supplied via environment variables (UNBOXAI_API_KEY).

You aquire a API key by signing up [HERE](https://unboxai.com/behaviorgpt)

## Imports

In [ ]:
from behaviorgpt import UnboxAIClient, Search, View, AddToCart, Order

# To load API Key from .env
from dotenv import load_dotenv
load_dotenv(override=True);

## Interacting with BehaviorGPT

Your code communicates with our services through an API client. This client packages your structured history and query parameters into standard HTTP requests dispatched to our servers.

In [ ]:
client = UnboxAIClient(market="us", default_catalog_id="amazon_catalog")

Here we are browsing a sample E-Com Marketplace catalog as a user from the `USA`

### Key Concepts

A quick breakdown of the core terms used in this notebook:

* **Event:** A single action taken by the shopper, such as viewing a product, adding to cart, or typing a query.
* **History:** The chronological sequence of *events* in a user's session. This is the behavioral context fed into the AI.
* **Cold Start:** A scenario where a user has absolutely no history. The engine must recommend products without any personalization signals.
* **Score:** The numerical weight the engine calculates for a product to determine its ranking and relevance to the current user.
* **Vocabulary:** The catalog of known terms, attributes, and entities the engine uses to map user intent to actual products.

# Hello, BehaviorGPT

Before anything, here is an example of the model answering a query.

In [ ]:
res = client.complete(history=[Search("lego")], limit=5)
res.to_pandas()[["name", "price", "score"]]

In the code-cell; `complete` runs a search in the given catalog for "lego"

In the output; `score` is the model’s confidence.

## What the model can return

Every product in the catalog has a position in a space the model learned. That position comes from how people behaved around it — who viewed, what they viewed next, what ended up in the same basket. Two products sit close together when people treat them similarly.

The catalog is the model’s vocabulary; the complete list of things it can return. Its job is to pick which one comes next given a sequence of actions. Nothing outside the vocabulary can ever come back.

In [ ]:
# Example known product_ids from the catalog in use by default

NIKE_PANTS = "B08NYK61PJ"

EXPENSIVE_SNEAKERS_1 = "B09DBFS1Y5"
EXPENSIVE_SNEAKERS_2 = "B0BS1YV27T"

CHEAP_SNEAKERS_1 = "B0CNFSW297"
CHEAP_SNEAKERS_2 = "B08P34GCHQ"

## One call does everything

**BehaviorGPT** does one thing: given a sequence of events, predict what comes next.

- A search where the last event is a query.
- A recommendation  where the last event is a product.

Different features in a store — search, “more like this”, a homepage for a returning visitor — are the same call with different histories.

`complete` as in completing a sequence, the way a language model completes a sentence.

In [ ]:
scenarios = {
    "cold start":          [],
    "search":              [Search("shoes")],
    "more like this":      [View(NIKE_PANTS)],
    "personalized search": [View(NIKE_PANTS), Search("shoes")],
    "post-purchase":       [View(NIKE_PANTS), AddToCart(NIKE_PANTS), Order()],
}

for label, history in scenarios.items():
    top = client.complete(history=history, limit=3)
    print(f"{label:22} -> {', '.join(top.names)}")

## Context changes the answer

Same final query, two different preceding events:

In [ ]:
res_one = client.complete(history=[Search("nike"), Search("shoes")], limit=5)
res_two = client.complete(history=[Search("adidas"), Search("shoes")], limit=5)

display(res_one.to_pandas()[["name", "price", "score"]], res_two.to_pandas()[["name", "price", "score"]])

The word "shoes" didn’t change, what came before it did. The model reads the whole sequence, not just the last thing typed.

In [ ]:
QUERY = "sneakers"

for label, history in {
    "no history": [],
    "expensive":  [View(EXPENSIVE_SNEAKERS_1), AddToCart(EXPENSIVE_SNEAKERS_1), View(EXPENSIVE_SNEAKERS_2), AddToCart(EXPENSIVE_SNEAKERS_2)],
    "budget":     [View(CHEAP_SNEAKERS_1), AddToCart(CHEAP_SNEAKERS_1), View(CHEAP_SNEAKERS_2), AddToCart(CHEAP_SNEAKERS_2)],
}.items():
    res = client.complete(history=[*history, Search(QUERY)], catalog_id="amazon_catalog", limit=5)

    print(f"{label:12} avg ${res.mean_price:>6.0f}  ->  {', '.join(res.names[:5])}")

The model picked up price sensitivity from two clicks, without being told anything about price.

# Bring your own catalog

The model can only return products from its vocabulary. Try using it on your store & embed your catalog! 

Upload a parquet file and an embed job places every product in the model's latent space. It requires your personal UnboxAI API key (from your invite email) in `.env` as `UNBOX_API_KEY`.

Read instructions on how to structure the file in the `/docs` folder. 

In [ ]:
CATALOG_PATH = "./sample_catalog.parquet"

job = client.embed(CATALOG_PATH, wait=True, timeout=1800.0)
job.model_dump()

## Embedding Space

Embedding your catalog converts each product into a vector, placing it in a shared vector space.

- **Placement:** Each product's position is derived from its data (titles, descriptions, attributes).
- **Proximity:** Similar products land close together; unrelated ones land far apart.
- **Retrieval:** Recommendations and search become nearest-neighbor lookups: find the vectors closest to a given point.

In [ ]:
from IPython.display import HTML
HTML(client.umap(catalog_id=job.catalog_id))

Check the accuracy of the space by finding products similar to a random one. To test a specific product, replace `random_product` with its id as a string.

In [ ]:
pick = client.random_product(job.catalog_id)
print(pick.data["name"])

response = client.similar_products(pick.id, catalog_id=job.catalog_id)
response.to_pandas()[["name", "price", "score"]]

The upload returns immediately while processing runs in the background: images are fetched, products are embedded, and the results are indexed. Until this finishes (a few minutes), queries return `status`. The returned `catalog_id` is private to your API key. `catalog_name` is your file name, for identifying the upload.

In [ ]:
res = client.complete(history=[Search("lego sets")], catalog_id=job.catalog_id, limit=5)
res.to_pandas()[["name", "price", "score"]]

In [ ]:
SOME_PRODUCT = res.products.items[0].data["id"]

scenarios = {
    "cold start":          [],
    "search":              [Search("kitchen")],
    "more like this":      [View(SOME_PRODUCT)],
    "personalized search": [View(SOME_PRODUCT), Search("kitchen")],
}

for label, history in scenarios.items():
    top = client.complete(catalog_id=job.catalog_id, history=history, limit=3)
    names = [item.data.get("name") for item in top.products.items]
    print(f"{label:22} -> {', '.join(names)}")

Once the catalog is embedded and all cells have run, try playing around with the catalog in the demo with BehaviorGPT as the recommendation engine:

1. Open [behaviorgpt.unboxai.com](https://behaviorgpt.unboxai.com/).
2. Select the **BehaviorGPT V4.0-13B** model.
3. In the catalog dropdown, choose **Bring your own catalog** and enter the API key from your email.

OBS! For this to work properly the images for the product catalog need to be publicly reachable.